# CYGNSS inundation visualization with land cover data in a form of split map

Import the necessary libraries for working with Google Earth Engine (GEE) and geemap.

In [1]:
import ee
import geemap
import matplotlib as mpl
import colormaps as cmaps
import cmocean
import numpy as np

SET PROJECT NAME, authenticate and set up your Earth Engine environment by creating a Google Cloud Project and enabling the Earth Engine API (https://www.google.com/url?q=https%3A%2F%2Fconsole.cloud.google.com%2Fapis%2Fapi%2Fearthengine.googleapis.com), if you haven’t already done so. Detailed setup instructions are available at https://developers.google.com/earth-engine/guides/access#a-role-in-a-cloud-project

In [2]:
project_name = 'YOUR_EE_PROJECTNAME'
ee.Authenticate()
ee.Initialize(project=project_name) 

Load the CYGNSS collection

In [3]:
cygnss_collection = ee.ImageCollection("projects/sat-io/open-datasets/CYGNSS/inundation")


Set up the functions to (i) remove NaN values from CYGNSS data and (ii) convert inundation data to fractional inundation

In [4]:
def mask_inundation(image):
    image = image.updateMask(image.neq(255))  # mask no-data values (255)
    return image.divide(100)  # scale to fractional values

Filter the CYGNSS collection by the period of interest, select the desired band, apply a function to mask NoData values and obtain fractional inundation, and calculate the mean value over the selected period.

In [5]:
mean_inundation_maps = (
    cygnss_collection
    .filterDate(ee.Date('2025-03-01'), ee.Date('2025-03-14'))
    .select('inundation_3day')
    .map(mask_inundation)
    .mean()
)


Select the land cover dataset and create an Image

In [6]:
worldcover_dataset = ee.ImageCollection('ESA/WorldCover/v200')

worldcover = ee.Image('ESA/WorldCover/v200/2021')

Set the visualization parameters (colors and value ranges) for the plotted band. You can define your own colormap. By default, GEE accepts HEX or CSS color codes. Separate visualization parameters are defined for inundation data (nominal range: 0 to 1) and land cover data.

In [7]:

# Inundation palette and visualisation params
cmap_inundation = mpl.colormaps["gist_earth_r"].resampled(256) # palette (gist_earth) from matplotlib (https://matplotlib.org/stable/users/explain/colors/colormaps.html)
palette_inundation_hex = [mpl.colors.rgb2hex(cmap_inundation(i)) for i in range(256)]

inundation_vis = {
    'min': 0,
    'max': 1,
    'palette': palette_inundation_hex
}


#Cropland classification
worldcover_vis = {
    'min': 10,
    'max': 100,
    'palette': [
        '#006400',  # Trees
        '#ffbb22',  # Shrubland
        '#ffff4c',  # Grassland
        '#f096ff',  # Cropland
        '#fa0000',  # Built-up
        '#b4b4b4',  # Bare / sparse
        '#f0f0f0',  # Snow / ice
        '#0064c8',  # Water
        '#0096a0',  # Wetland
        '#00cf75',  # Mangroves
        '#fae6a0'   # Moss / lichen
    ]
}

Plot the region of interest with inundation and land cover data.

In [8]:
# Region of interest – South Sudan
roi = ee.Geometry.Rectangle([
    33, 8,   # west, south
    28, 13     # east, north
])

Map = geemap.Map() # generate map
Map.add_basemap('HYBRID') # add the 'HYBRID' layer containing satellite imagery
left_layer = geemap.ee_tile_layer(mean_inundation_maps, inundation_vis, "Fractional Inundation") # Create left layer of split map with fractional inundation
right_layer = geemap.ee_tile_layer(worldcover, worldcover_vis, "Cropland (ESA WorldCover)", opacity=0.6) ## Create right layer of split map with land cover
Map.split_map(left_layer, right_layer, left_label='Inundation', right_label='Landcover') # Add left and right layer to the map
Map.add_colorbar(inundation_vis, label="Inundation Fraction", orientation="vertical", layer_name="Inundation", position="bottomleft") # Add colorbar to the layer with inundation data
#create legend for a land cover data
legend_dict = {
    "Trees": "#006400",
    "Shrubland": "#ffbb22",
    "Grassland": "#ffff4c",
    "Cropland": "#f096ff",
    "Built-up": "#fa0000",
    "Bare / Sparse": "#b4b4b4",
    "Snow / Ice": "#f0f0f0",
    "Water": "#0064c8",
    "Wetland": "#0096a0",
    "Mangroves": "#00cf75",
    "Moss / Lichen": "#fae6a0",
}
Map.add_legend(title="Land Cover", legend_dict=legend_dict, position= "bottomright") # Add colorbar to the layer with land cover data
Map.center_object(roi, 7) # Set the initial view to the ROI
Map # Display the map

Map(center=[10.503000482394638, 30.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…